In [1]:
# CELL 1 — QWEN2.5-3B EXPERIMENT SETUP

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import torch

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

ROOT = Path("/content/drive/MyDrive/slm-distillation")
RUN = ROOT / "outputs" / "qwen25_3b"

RUN.mkdir(parents=True, exist_ok=True)

print("===== QWEN2.5-3B EXPERIMENT =====")
print("Model:", MODEL_ID)
print("Output:", RUN)

print("\n===== GPU =====")

if torch.cuda.is_available():
    print("✅ GPU:", torch.cuda.get_device_name(0))
    print(
        "✅ VRAM:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )
else:
    print("❌ No GPU detected")

print("\n===== DATA =====")

for name in ["train.jsonl", "val.jsonl", "test.jsonl"]:
    path = ROOT / "data" / "processed" / name

    print(
        "✅" if path.exists() else "❌",
        name,
        path
    )

Mounted at /content/drive
===== QWEN2.5-3B EXPERIMENT =====
Model: Qwen/Qwen2.5-3B-Instruct
Output: /content/drive/MyDrive/slm-distillation/outputs/qwen25_3b

===== GPU =====
✅ GPU: Tesla T4
✅ VRAM: 14.56 GB

===== DATA =====
✅ train.jsonl /content/drive/MyDrive/slm-distillation/data/processed/train.jsonl
✅ val.jsonl /content/drive/MyDrive/slm-distillation/data/processed/val.jsonl
✅ test.jsonl /content/drive/MyDrive/slm-distillation/data/processed/test.jsonl


In [3]:
# CELL 2 — DEPENDENCIES + HUGGING FACE ACCESS CHECK

import sys
import subprocess
import os
import torch

print("🔧 Installing/checking packages...")

packages = [
    "transformers>=4.45,<5",
    "accelerate>=1.0",
    "peft>=0.13",
    "bitsandbytes>=0.46.1",
    "datasets>=3.0",
    "huggingface_hub>=0.25",
]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + packages,
    check=True
)

print("✅ Packages ready")


# ------------------------------------------------------------
# Hugging Face token
# ------------------------------------------------------------

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("✅ HF_TOKEN found")
else:
    print("⚠️ HF_TOKEN not found")


# ------------------------------------------------------------
# Check Qwen model access
# ------------------------------------------------------------

from huggingface_hub import HfApi

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

try:
    api = HfApi(token=HF_TOKEN)
    info = api.model_info(MODEL_ID)

    print("\n===== MODEL ACCESS =====")
    print("✅ Model found:", info.id)
    print("✅ Access check passed")

except Exception as e:
    print("\n❌ Model access problem:")
    print(type(e).__name__, str(e))


# ------------------------------------------------------------
# GPU check
# ------------------------------------------------------------

print("\n===== GPU =====")
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("✅ GPU:", torch.cuda.get_device_name(0))

🔧 Installing/checking packages...
✅ Packages ready
✅ HF_TOKEN found

===== MODEL ACCESS =====
✅ Model found: Qwen/Qwen2.5-3B-Instruct
✅ Access check passed

===== GPU =====
CUDA available: True
✅ GPU: Tesla T4


In [4]:
# CELL 2 — DEPENDENCIES + HUGGING FACE ACCESS CHECK

import sys
import subprocess
import os
import torch

print("🔧 Installing/checking packages...")

packages = [
    "transformers>=4.45,<5",
    "accelerate>=1.0",
    "peft>=0.13",
    "bitsandbytes>=0.46.1",
    "datasets>=3.0",
    "huggingface_hub>=0.25",
]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + packages,
    check=True
)

print("✅ Packages ready")


# ------------------------------------------------------------
# Hugging Face token
# ------------------------------------------------------------

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("✅ HF_TOKEN found")
else:
    print("⚠️ HF_TOKEN not found")


# ------------------------------------------------------------
# Check Qwen model access
# ------------------------------------------------------------

from huggingface_hub import HfApi

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

try:
    api = HfApi(token=HF_TOKEN)
    info = api.model_info(MODEL_ID)

    print("\n===== MODEL ACCESS =====")
    print("✅ Model found:", info.id)
    print("✅ Access check passed")

except Exception as e:
    print("\n❌ Model access problem:")
    print(type(e).__name__, str(e))


# ------------------------------------------------------------
# GPU check
# ------------------------------------------------------------

print("\n===== GPU =====")
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("✅ GPU:", torch.cuda.get_device_name(0))

🔧 Installing/checking packages...
✅ Packages ready
✅ HF_TOKEN found

===== MODEL ACCESS =====
✅ Model found: Qwen/Qwen2.5-3B-Instruct
✅ Access check passed

===== GPU =====
CUDA available: True
✅ GPU: Tesla T4


In [5]:
# CELL 3 — LOAD QWEN2.5-3B IN 4-BIT

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

print("🔄 Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("✅ Tokenizer loaded")
print("Chat template available:", tokenizer.chat_template is not None)


# ------------------------------------------------------------
# 4-bit QLoRA configuration
# ------------------------------------------------------------

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

print("\n🔄 Loading Qwen2.5-3B in 4-bit...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

model.config.use_cache = False

print("\n==============================")
print("✅ QWEN LOADED SUCCESSFULLY")
print("==============================")

print("Model:", MODEL_ID)
print("Device:", next(model.parameters()).device)
print("4-bit:", getattr(model, "is_loaded_in_4bit", False))

print(
    "Allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

print(
    "Reserved:",
    round(torch.cuda.memory_reserved() / 1024**3, 2),
    "GB"
)

print(
    "Total GPU:",
    round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
    "GB"
)

🔄 Loading tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


✅ QWEN LOADED SUCCESSFULLY
Model: Qwen/Qwen2.5-3B-Instruct
Device: cuda:0
4-bit: True
Allocated: 1.91 GB
Reserved: 2.94 GB
Total GPU: 14.56 GB


In [6]:
# CELL 4 — FORMAT PROJECT DATA FOR QWEN + VERIFY SYSTEM PROMPT

import json
import re
import statistics
from pathlib import Path
from collections import Counter
from datasets import Dataset

ROOT = Path("/content/drive/MyDrive/slm-distillation")
DATA = ROOT / "data" / "processed"
RUN = ROOT / "outputs" / "qwen25_3b"

RUN.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 1. Parse original project ChatML rows
# ------------------------------------------------------------

def parse_project_example(row):
    text = row["text"]

    system_match = re.search(
        r"<\|im_start\|>system\n(.*?)<\|im_end\|>",
        text,
        re.DOTALL,
    )

    user_match = re.search(
        r"<\|im_start\|>user\n(.*?)<\|im_end\|>",
        text,
        re.DOTALL,
    )

    assistant_match = re.search(
        r"<\|im_start\|>assistant\n(.*?)<\|im_end\|>",
        text,
        re.DOTALL,
    )

    if not user_match or not assistant_match:
        raise ValueError("Could not parse project example")

    return {
        "system": system_match.group(1).strip() if system_match else "",
        "user": user_match.group(1).strip(),
        "assistant": assistant_match.group(1).strip(),
        "cluster_id": row["cluster_id"],
        "prompt_id": row["prompt_id"],
    }


# ------------------------------------------------------------
# 2. Format with Qwen native chat template
# ------------------------------------------------------------

def format_for_qwen(parsed):

    messages = [
        {
            "role": "system",
            "content": parsed["system"],
        },
        {
            "role": "user",
            "content": parsed["user"],
        },
        {
            "role": "assistant",
            "content": parsed["assistant"],
        },
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )


# ------------------------------------------------------------
# 3. Load train / val / test
# ------------------------------------------------------------

def load_split(filename):

    rows = []

    with open(DATA / filename, "r", encoding="utf-8") as f:

        for line in f:

            if not line.strip():
                continue

            original = json.loads(line)
            parsed = parse_project_example(original)

            rows.append({
                "text": format_for_qwen(parsed),
                "cluster_id": parsed["cluster_id"],
                "prompt_id": parsed["prompt_id"],
                "expected": parsed["assistant"],
                "user_prompt": parsed["user"],
                "system_prompt": parsed["system"],
            })

    return rows


train_rows = load_split("train.jsonl")
val_rows = load_split("val.jsonl")
test_rows = load_split("test.jsonl")

train_dataset = Dataset.from_list(train_rows)
val_dataset = Dataset.from_list(val_rows)
test_dataset = Dataset.from_list(test_rows)


# ------------------------------------------------------------
# 4. Dataset checks
# ------------------------------------------------------------

print("===== QWEN DATASET =====")

print("Train:", len(train_dataset))
print("Val:", len(val_dataset))
print("Test:", len(test_dataset))

print(
    "Train prompt IDs:",
    dict(Counter(train_dataset["prompt_id"]))
)


# ------------------------------------------------------------
# 5. Token lengths
# ------------------------------------------------------------

lengths = []

for text in train_dataset["text"]:

    ids = tokenizer(
        text,
        add_special_tokens=False,
    )["input_ids"]

    lengths.append(len(ids))


print("\n===== TOKEN LENGTHS =====")

print("Min:", min(lengths))
print("Median:", int(statistics.median(lengths)))
print("Max:", max(lengths))


# ------------------------------------------------------------
# 6. Explicitly verify system prompt survived
# ------------------------------------------------------------

first_text = train_dataset[0]["text"]

system_preserved = (
    "You are an expert at analyzing IT, HR, and customer experience support tickets"
    in first_text
)

print("\nSystem prompt preserved:", system_preserved)


# ------------------------------------------------------------
# 7. Show first exact Qwen training example
# ------------------------------------------------------------

print("\n===== FIRST QWEN EXAMPLE =====")
print(first_text)

print("\nExpected label:")
print(train_dataset[0]["expected"])


# ------------------------------------------------------------
# 8. Save starting experiment config
# ------------------------------------------------------------

config = {
    "model_id": MODEL_ID,
    "method": "4-bit QLoRA",
    "prompt_format": "Qwen native chat template",
    "system_prompt_strategy": "native system role",
    "system_prompt_preserved": system_preserved,
    "train_examples": len(train_dataset),
    "val_examples": len(val_dataset),
    "test_examples": len(test_dataset),
    "min_train_tokens": min(lengths),
    "median_train_tokens": int(statistics.median(lengths)),
    "max_train_tokens": max(lengths),
}

config_file = RUN / "experiment_config.json"

with open(config_file, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print("\n✅ Config saved:")
print(config_file)

===== QWEN DATASET =====
Train: 70
Val: 15
Test: 20
Train prompt IDs: {'P1': 14, 'P2': 14, 'P3': 14, 'P4': 14, 'P5': 14}

===== TOKEN LENGTHS =====
Min: 238
Median: 261
Max: 309

System prompt preserved: True

===== FIRST QWEN EXAMPLE =====
<|im_start|>system
You are an expert at analyzing IT, HR, and customer experience support tickets. Your task is to generate a concise, descriptive label for a cluster of support tickets. The label should capture the dominant theme, be specific enough to distinguish this cluster from others, and be 5-15 words long. Return only the label — no explanation, no punctuation at the end, no quotes.<|im_end|>
<|im_start|>user
Generate a concise label (5-15 words) for this cluster of support tickets.

Ticket 1: I need assistance to see when will my article arrive
Ticket 2: I have to check when my package is going to arrive
Ticket 3: can you show me when my product is going to arrive?
Ticket 4: I want to see how long it tyakes for my package to arrive
Ticket 5

In [7]:
# CELL 5 — QWEN2.5-3B BASELINE EVALUATION

import torch
import pandas as pd
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/slm-distillation")
RUN = ROOT / "outputs" / "qwen25_3b"

print("🔄 Running Qwen2.5-3B baseline on 20 test examples...\n")

model.eval()
model.config.use_cache = True

baseline_results = []

for i, row in enumerate(test_rows, start=1):

    messages = [
        {
            "role": "system",
            "content": row["system_prompt"],
        },
        {
            "role": "user",
            "content": row["user_prompt"],
        },
    ]

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
        add_special_tokens=False,
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=40,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    prediction = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    ).strip()

    baseline_results.append({
        "example": i,
        "cluster_id": row["cluster_id"],
        "prompt_id": row["prompt_id"],
        "baseline_prediction": prediction,
        "expected": row["expected"],
    })

    print(
        f"{i:02d}/20 | "
        f"{row['prompt_id']} | "
        f"{prediction}"
    )


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

baseline_df = pd.DataFrame(baseline_results)

baseline_file = RUN / "baseline_test_predictions.csv"

baseline_df.to_csv(
    baseline_file,
    index=False
)

model.config.use_cache = False

print("\n==============================")
print("QWEN BASELINE COMPLETE")
print("==============================")

print("Examples:", len(baseline_df))

print("\n✅ Saved:")
print(baseline_file)

print("\n✅ Ready for QLoRA setup")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


🔄 Running Qwen2.5-3B baseline on 20 test examples...

01/20 | P1 | Shipment Options Inquiry
02/20 | P2 | Shipment Options Inquiry
03/20 | P3 | Shipment Options Inquiry Requests
04/20 | P4 | Shipment Options Inquiry Support
05/20 | P5 | Shipment Options Inquiry
06/20 | P1 | Notify Sign-Up Issues Errors Problems Inform
07/20 | P2 | notify sign-up issues errors problems notify signup
08/20 | P3 | Sign-up notification issues
09/20 | P4 | Notify sign-up issues errors problems
10/20 | P5 | notify sign-up issues errors problems notify signup errors
11/20 | P1 | PIN Reset Assistance Requests
12/20 | P2 | PIN Reset And Recovery Requests
13/20 | P3 | PIN Reset And Recovery Requests
14/20 | P4 | PIN Reset And Recovery Requests
15/20 | P5 | PIN reset and recovery inquiries
16/20 | P1 | Switching gold account issues
17/20 | P2 | Switching gold account issues
18/20 | P3 | Switching Modifying Gold Account Info
19/20 | P4 | Switching Modifying Gold Account Info
20/20 | P5 | switching gold account requ

In [8]:
# CELL 6 — PREPARE QWEN2.5-3B FOR QLoRA

import json
import torch
from pathlib import Path

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

ROOT = Path("/content/drive/MyDrive/slm-distillation")
RUN = ROOT / "outputs" / "qwen25_3b"

print("🔧 Preparing Qwen2.5-3B for QLoRA...")

# ------------------------------------------------------------
# 1. Prepare quantized model for training
# ------------------------------------------------------------

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True
)

# ------------------------------------------------------------
# 2. LoRA configuration
# Same setup as Mistral for fair comparison
# ------------------------------------------------------------

lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

model = get_peft_model(
    model,
    lora_config
)

# Keep trainable LoRA parameters FP32
for param in model.parameters():
    if param.requires_grad:
        param.data = param.data.float()

model.config.use_cache = False

# ------------------------------------------------------------
# 3. Check trainable params
# ------------------------------------------------------------

print("\n===== QWEN LORA CHECK =====")

model.print_trainable_parameters()

trainable_dtypes = {
    str(p.dtype)
    for p in model.parameters()
    if p.requires_grad
}

print("Trainable dtypes:", trainable_dtypes)

print(
    "GPU allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

print(
    "GPU reserved:",
    round(torch.cuda.memory_reserved() / 1024**3, 2),
    "GB"
)

# ------------------------------------------------------------
# 4. Update experiment config
# ------------------------------------------------------------

config_file = RUN / "experiment_config.json"

with open(config_file, "r", encoding="utf-8") as f:
    config = json.load(f)

config.update({
    "lora_r": 16,
    "lora_alpha": 16,
    "lora_dropout": 0.05,
    "target_modules": [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
})

with open(config_file, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print("\n✅ Config updated:")
print(config_file)

print("\n✅ Qwen ready for training")

🔧 Preparing Qwen2.5-3B for QLoRA...

===== QWEN LORA CHECK =====
trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607
Trainable dtypes: {'torch.float32'}
GPU allocated: 2.62 GB
GPU reserved: 4.22 GB

✅ Config updated:
/content/drive/MyDrive/slm-distillation/outputs/qwen25_3b/experiment_config.json

✅ Qwen ready for training


In [9]:
# CELL 7 — TRAIN QWEN2.5-3B WITH QLoRA ON T4

import json
import gc
import torch
from pathlib import Path

from transformers import (
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
)

ROOT = Path("/content/drive/MyDrive/slm-distillation")
RUN = ROOT / "outputs" / "qwen25_3b"

TRAINING_DIR = RUN / "training"
FINAL_ADAPTER = RUN / "final_adapter"

TRAINING_DIR.mkdir(parents=True, exist_ok=True)
FINAL_ADAPTER.mkdir(parents=True, exist_ok=True)

MAX_LENGTH = 384


# ------------------------------------------------------------
# 1. Tokenize
# ------------------------------------------------------------

print("🔄 Tokenizing Qwen training data...")

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
        add_special_tokens=False,
    )

tokenized_train = train_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=train_dataset.column_names,
)

tokenized_val = val_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=val_dataset.column_names,
)

print("✅ Train:", len(tokenized_train))
print("✅ Val:", len(tokenized_val))


# ------------------------------------------------------------
# 2. Causal LM collator
# ------------------------------------------------------------

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)


# ------------------------------------------------------------
# 3. Training prep
# ------------------------------------------------------------

model.train()
model.config.use_cache = False
model.gradient_checkpointing_enable()

if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

for p in model.parameters():
    if p.requires_grad:
        p.data = p.data.float()

gc.collect()
torch.cuda.empty_cache()


# ------------------------------------------------------------
# 4. Training configuration
# Same main settings as Mistral
# ------------------------------------------------------------

training_args = TrainingArguments(
    output_dir=str(TRAINING_DIR),

    num_train_epochs=3,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    gradient_accumulation_steps=8,

    learning_rate=2e-4,
    warmup_ratio=0.1,

    logging_steps=1,

    eval_strategy="epoch",
    save_strategy="epoch",

    save_total_limit=3,

    # Keep Trainer AMP disabled because of the
    # previous Colab GradScaler issue.
    fp16=False,
    bf16=False,

    gradient_checkpointing=True,

    optim="adamw_torch",

    max_grad_norm=1.0,

    report_to="none",

    remove_unused_columns=False,

    seed=42,
)


# ------------------------------------------------------------
# 5. Trainer
# ------------------------------------------------------------

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)


# ------------------------------------------------------------
# 6. Show setup
# ------------------------------------------------------------

print("\n==============================")
print("QWEN TRAINING SETUP")
print("==============================")

print("Model:", MODEL_ID)
print("Train examples:", len(tokenized_train))
print("Validation examples:", len(tokenized_val))
print("Epochs:", training_args.num_train_epochs)
print("Batch size:", training_args.per_device_train_batch_size)
print("Gradient accumulation:", training_args.gradient_accumulation_steps)

print(
    "Effective batch size:",
    training_args.per_device_train_batch_size
    * training_args.gradient_accumulation_steps
)

print("Learning rate:", training_args.learning_rate)
print("Max sequence length:", MAX_LENGTH)

print("Trainer FP16:", training_args.fp16)
print("Trainer BF16:", training_args.bf16)

print(
    "GPU allocated before training:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

print(
    "GPU reserved before training:",
    round(torch.cuda.memory_reserved() / 1024**3, 2),
    "GB"
)


# ------------------------------------------------------------
# 7. TRAIN
# ------------------------------------------------------------

print("\n🚀 STARTING QWEN QLoRA TRAINING\n")

train_result = trainer.train()


# ------------------------------------------------------------
# 8. Save epoch-3 adapter
# ------------------------------------------------------------

model.save_pretrained(FINAL_ADAPTER)
tokenizer.save_pretrained(FINAL_ADAPTER)


# ------------------------------------------------------------
# 9. Save training metrics
# ------------------------------------------------------------

metrics = dict(train_result.metrics)

metrics.update({
    "model_id": MODEL_ID,
    "max_length": MAX_LENGTH,
    "epochs": 3,
    "learning_rate": 2e-4,
    "train_batch_size": 1,
    "gradient_accumulation_steps": 8,
})

metrics_file = RUN / "training_metrics.json"

with open(metrics_file, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)


# ------------------------------------------------------------
# 10. Final validation
# ------------------------------------------------------------

print("\n🔄 Final validation evaluation...")

eval_metrics = trainer.evaluate()

eval_file = RUN / "validation_metrics.json"

with open(eval_file, "w", encoding="utf-8") as f:
    json.dump(eval_metrics, f, indent=2)


# ------------------------------------------------------------
# Complete
# ------------------------------------------------------------

print("\n==============================")
print("✅ QWEN TRAINING COMPLETE")
print("==============================")

print("\nFinal adapter:")
print(FINAL_ADAPTER)

print("\nTraining metrics:")
print(metrics)

print("\nValidation metrics:")
print(eval_metrics)

print("\n✅ Saved:")
print(metrics_file)
print(eval_file)

🔄 Tokenizing Qwen training data...


Map:   0%|          | 0/70 [00:00<?, ? examples/s]

Map:   0%|          | 0/15 [00:00<?, ? examples/s]

✅ Train: 70
✅ Val: 15

QWEN TRAINING SETUP
Model: Qwen/Qwen2.5-3B-Instruct
Train examples: 70
Validation examples: 15
Epochs: 3
Batch size: 1
Gradient accumulation: 8
Effective batch size: 8
Learning rate: 0.0002
Max sequence length: 384
Trainer FP16: False
Trainer BF16: False
GPU allocated before training: 2.62 GB
GPU reserved before training: 4.22 GB

🚀 STARTING QWEN QLoRA TRAINING



Epoch,Training Loss,Validation Loss
1,1.784200,1.532189
2,1.045800,0.995797
3,0.974700,0.878947



🔄 Final validation evaluation...



✅ QWEN TRAINING COMPLETE

Final adapter:
/content/drive/MyDrive/slm-distillation/outputs/qwen25_3b/final_adapter

Training metrics:
{'train_runtime': 213.8052, 'train_samples_per_second': 0.982, 'train_steps_per_second': 0.126, 'total_flos': 928515189178368.0, 'train_loss': 1.5123535814108673, 'epoch': 3.0, 'model_id': 'Qwen/Qwen2.5-3B-Instruct', 'max_length': 384, 'epochs': 3, 'learning_rate': 0.0002, 'train_batch_size': 1, 'gradient_accumulation_steps': 8}

Validation metrics:
{'eval_loss': 0.8789467215538025, 'eval_runtime': 4.4701, 'eval_samples_per_second': 3.356, 'eval_steps_per_second': 3.356, 'epoch': 3.0}

✅ Saved:
/content/drive/MyDrive/slm-distillation/outputs/qwen25_3b/training_metrics.json
/content/drive/MyDrive/slm-distillation/outputs/qwen25_3b/validation_metrics.json


In [10]:
# CELL 8 — CHECK QWEN CHECKPOINTS + SAVE TRAINING SUMMARY

from pathlib import Path
import json

ROOT = Path("/content/drive/MyDrive/slm-distillation")
RUN = ROOT / "outputs" / "qwen25_3b"
TRAINING_DIR = RUN / "training"

print("===== SAVED CHECKPOINTS =====")

checkpoints = sorted(
    [p for p in TRAINING_DIR.glob("checkpoint-*") if p.is_dir()],
    key=lambda p: int(p.name.split("-")[-1])
)

for checkpoint in checkpoints:
    print("✅", checkpoint)

print("\n===== VALIDATION LOSSES =====")
print("Epoch 1: 1.532189")
print("Epoch 2: 0.995797")
print("Epoch 3: 0.878947  <-- BEST")

best_checkpoint = TRAINING_DIR / "checkpoint-27"

print("\nBest checkpoint:")
print(best_checkpoint)

print("\nExists:", best_checkpoint.exists())

# Save conclusion
summary = {
    "model": "Qwen/Qwen2.5-3B-Instruct",
    "training_status": "successful",
    "epochs": 3,
    "train_loss": 1.5123535814108673,
    "validation_loss_epoch_1": 1.532189,
    "validation_loss_epoch_2": 0.995797,
    "validation_loss_epoch_3": 0.8789467215538025,
    "best_epoch": 3,
    "best_validation_loss": 0.8789467215538025,
    "best_checkpoint": str(best_checkpoint),
    "observation": (
        "Validation loss improved across all three epochs, "
        "with epoch 3 producing the best validation loss."
    )
}

summary_file = RUN / "training_summary.json"

with open(summary_file, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("\n✅ Training summary saved:")
print(summary_file)

===== SAVED CHECKPOINTS =====
✅ /content/drive/MyDrive/slm-distillation/outputs/qwen25_3b/training/checkpoint-9
✅ /content/drive/MyDrive/slm-distillation/outputs/qwen25_3b/training/checkpoint-18
✅ /content/drive/MyDrive/slm-distillation/outputs/qwen25_3b/training/checkpoint-27

===== VALIDATION LOSSES =====
Epoch 1: 1.532189
Epoch 2: 0.995797
Epoch 3: 0.878947  <-- BEST

Best checkpoint:
/content/drive/MyDrive/slm-distillation/outputs/qwen25_3b/training/checkpoint-27

Exists: True

✅ Training summary saved:
/content/drive/MyDrive/slm-distillation/outputs/qwen25_3b/training_summary.json


In [11]:
# CELL 9 — EVALUATE BEST FINE-TUNED QWEN CHECKPOINT

import torch
import pandas as pd
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/slm-distillation")
RUN = ROOT / "outputs" / "qwen25_3b"

BEST_CHECKPOINT = RUN / "training" / "checkpoint-27"
SAVE_FILE = RUN / "finetuned_test_predictions.csv"

print("===== BEST CHECKPOINT =====")
print(BEST_CHECKPOINT)
print("Exists:", BEST_CHECKPOINT.exists())

# ------------------------------------------------------------
# 1. Load best epoch-3 adapter
# ------------------------------------------------------------

ADAPTER_NAME = "best_epoch3"

if ADAPTER_NAME not in model.peft_config:
    model.load_adapter(
        str(BEST_CHECKPOINT),
        adapter_name=ADAPTER_NAME
    )

model.set_adapter(ADAPTER_NAME)

print("✅ Best epoch-3 adapter loaded")


# ------------------------------------------------------------
# 2. Inference mode
# ------------------------------------------------------------

model.eval()
model.gradient_checkpointing_disable()
model.config.use_cache = True

finetuned_results = []

print("\n🔄 Running fine-tuned Qwen on 20 test examples...\n")


# ------------------------------------------------------------
# 3. Same exact test set
# ------------------------------------------------------------

for i, row in enumerate(test_rows, start=1):

    messages = [
        {
            "role": "system",
            "content": row["system_prompt"],
        },
        {
            "role": "user",
            "content": row["user_prompt"],
        },
    ]

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
        add_special_tokens=False,
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=40,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    prediction = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    ).strip()

    finetuned_results.append({
        "example": i,
        "cluster_id": row["cluster_id"],
        "prompt_id": row["prompt_id"],
        "prediction": prediction,
        "expected": row["expected"],
    })

    print(
        f"{i:02d}/20 | "
        f"{row['prompt_id']} | "
        f"{prediction}"
    )


# ------------------------------------------------------------
# 4. Save
# ------------------------------------------------------------

finetuned_df = pd.DataFrame(
    finetuned_results
)

finetuned_df.to_csv(
    SAVE_FILE,
    index=False
)

print("\n==============================")
print("QWEN FINE-TUNED EVALUATION COMPLETE")
print("==============================")

print("Checkpoint: epoch 3 / checkpoint-27")
print("Examples:", len(finetuned_df))

print("\n✅ Saved:")
print(SAVE_FILE)

display(
    finetuned_df[
        [
            "example",
            "prompt_id",
            "prediction",
            "expected",
        ]
    ]
)

===== BEST CHECKPOINT =====
/content/drive/MyDrive/slm-distillation/outputs/qwen25_3b/training/checkpoint-27
Exists: True
✅ Best epoch-3 adapter loaded

🔄 Running fine-tuned Qwen on 20 test examples...

01/20 | P1 | Customer seeking information on available shipping/delivery methods
02/20 | P2 | Customer seeking information on available shipping and delivery methods
03/20 | P3 | Customer Inquiry: Available Shipping and Delivery Methods
04/20 | P4 | Customer seeking information on available shipping and delivery methods
05/20 | P5 | Shipment and Delivery Options Inquiry
06/20 | P1 | Sign-up Assistance and Error Reporting
07/20 | P2 | User seeking assistance with reporting sign-up errors or issues
08/20 | P3 | Sign-up Process Assistance and Error Reporting
09/20 | P4 | User seeking assistance to report or inform about sign-up-related issues
10/20 | P5 | User seeking assistance to report or inform about sign-up-related issues or errors
11/20 | P1 | PIN Reset Assistance and Recovery Suppor

,example,prompt_id,prediction,expected
0,1,P1,Customer seeking information on available ship...,Inquiries about available shipping and deliver...
1,2,P2,Customer seeking information on available ship...,Inquiry about available shipping and delivery ...
2,3,P3,Customer Inquiry: Available Shipping and Deliv...,Shipping and Delivery Options Inquiry
3,4,P4,Customer seeking information on available ship...,Customers requesting information about availab...
4,5,P5,Shipment and Delivery Options Inquiry,Shipping and delivery options inquiry
5,6,P1,Sign-up Assistance and Error Reporting,Sign-up process errors and account creation fa...
6,7,P2,User seeking assistance with reporting sign-up...,Sign-up process errors and notification assist...
7,8,P3,Sign-up Process Assistance and Error Reporting,Sign-up and Registration Error Reporting
8,9,P4,User seeking assistance to report or inform ab...,Users seeking assistance reporting sign-up and...
9,10,P5,User seeking assistance to report or inform ab...,Sign-up registration errors and problem reporting


In [12]:
# CELL 10 — QWEN BASELINE VS FINE-TUNED COSINE SIMILARITY

import sys
import subprocess
import pandas as pd
from pathlib import Path

# Install sentence-transformers if needed
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"],
    check=True
)

from sentence_transformers import SentenceTransformer

ROOT = Path("/content/drive/MyDrive/slm-distillation")
RUN = ROOT / "outputs" / "qwen25_3b"

baseline_file = RUN / "baseline_test_predictions.csv"
finetuned_file = RUN / "finetuned_test_predictions.csv"

baseline = pd.read_csv(baseline_file)
finetuned = pd.read_csv(finetuned_file)

# Merge same 20 examples
df = finetuned.merge(
    baseline[
        [
            "example",
            "baseline_prediction"
        ]
    ],
    on="example"
)

print("🔄 Loading semantic similarity model...")

embedder = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

# ------------------------------------------------------------
# Embeddings
# ------------------------------------------------------------

expected_embeddings = embedder.encode(
    df["expected"].tolist(),
    normalize_embeddings=True
)

baseline_embeddings = embedder.encode(
    df["baseline_prediction"].tolist(),
    normalize_embeddings=True
)

finetuned_embeddings = embedder.encode(
    df["prediction"].tolist(),
    normalize_embeddings=True
)

# Cosine similarity
df["baseline_similarity"] = (
    baseline_embeddings * expected_embeddings
).sum(axis=1)

df["finetuned_similarity"] = (
    finetuned_embeddings * expected_embeddings
).sum(axis=1)

df["improvement"] = (
    df["finetuned_similarity"]
    - df["baseline_similarity"]
)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

baseline_avg = df["baseline_similarity"].mean()
finetuned_avg = df["finetuned_similarity"].mean()
avg_improvement = df["improvement"].mean()

better = (df["improvement"] > 0.0001).sum()
worse = (df["improvement"] < -0.0001).sum()
ties = len(df) - better - worse

print("\n==============================")
print("QWEN COSINE RESULTS")
print("==============================")

print(
    "Baseline average similarity:",
    round(baseline_avg, 4)
)

print(
    "Fine-tuned average similarity:",
    round(finetuned_avg, 4)
)

print(
    "Average improvement:",
    round(avg_improvement, 4)
)

print(
    f"Fine-tuned better on: {better}/{len(df)}"
)

print(
    f"Baseline better on: {worse}/{len(df)}"
)

print(
    f"Ties: {ties}/{len(df)}"
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

save_file = RUN / "baseline_vs_finetuned_comparison.csv"

df.to_csv(
    save_file,
    index=False
)

print("\n✅ Saved comparison:")
print(save_file)

display(
    df[
        [
            "example",
            "prompt_id",
            "baseline_prediction",
            "prediction",
            "expected",
            "baseline_similarity",
            "finetuned_similarity",
            "improvement",
        ]
    ]
)

🔄 Loading semantic similarity model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


QWEN COSINE RESULTS
Baseline average similarity: 0.7252
Fine-tuned average similarity: 0.8077
Average improvement: 0.0825
Fine-tuned better on: 17/20
Baseline better on: 3/20
Ties: 0/20

✅ Saved comparison:
/content/drive/MyDrive/slm-distillation/outputs/qwen25_3b/baseline_vs_finetuned_comparison.csv


,example,prompt_id,baseline_prediction,prediction,expected,baseline_similarity,finetuned_similarity,improvement
0,1,P1,Shipment Options Inquiry,Customer seeking information on available ship...,Inquiries about available shipping and deliver...,0.716786,0.857960,0.141175
1,2,P2,Shipment Options Inquiry,Customer seeking information on available ship...,Inquiry about available shipping and delivery ...,0.755765,0.824568,0.068803
2,3,P3,Shipment Options Inquiry Requests,Customer Inquiry: Available Shipping and Deliv...,Shipping and Delivery Options Inquiry,0.756692,0.827020,0.070329
3,4,P4,Shipment Options Inquiry Support,Customer seeking information on available ship...,Customers requesting information about availab...,0.660047,0.888316,0.228269
4,5,P5,Shipment Options Inquiry,Shipment and Delivery Options Inquiry,Shipping and delivery options inquiry,0.822996,0.925678,0.102681
5,6,P1,Notify Sign-Up Issues Errors Problems Inform,Sign-up Assistance and Error Reporting,Sign-up process errors and account creation fa...,0.743985,0.664276,-0.079709
6,7,P2,notify sign-up issues errors problems notify s...,User seeking assistance with reporting sign-up...,Sign-up process errors and notification assist...,0.782501,0.734023,-0.048478
7,8,P3,Sign-up notification issues,Sign-up Process Assistance and Error Reporting,Sign-up and Registration Error Reporting,0.589402,0.792637,0.203235
8,9,P4,Notify sign-up issues errors problems,User seeking assistance to report or inform ab...,Users seeking assistance reporting sign-up and...,0.583063,0.820835,0.237773
9,10,P5,notify sign-up issues errors problems notify s...,User seeking assistance to report or inform ab...,Sign-up registration errors and problem reporting,0.672953,0.730390,0.057436


In [13]:
# CELL 11 — QWEN LLM-AS-A-JUDGE

import sys
import subprocess
import os
import json
import random
import time
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# 1. Install Anthropic SDK if needed
# ------------------------------------------------------------

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "anthropic"],
    check=True
)

from anthropic import Anthropic
from google.colab import userdata


# ------------------------------------------------------------
# 2. API key
# ------------------------------------------------------------

ANTHROPIC_API_KEY = userdata.get("ANTHROPIC_API_KEY")

if not ANTHROPIC_API_KEY:
    raise RuntimeError(
        "ANTHROPIC_API_KEY not found in Colab Secrets"
    )

client = Anthropic(
    api_key=ANTHROPIC_API_KEY
)

print("✅ Anthropic API ready")


# ------------------------------------------------------------
# 3. Load Qwen comparison results
# ------------------------------------------------------------

ROOT = Path("/content/drive/MyDrive/slm-distillation")
RUN = ROOT / "outputs" / "qwen25_3b"

comparison_file = RUN / "baseline_vs_finetuned_comparison.csv"

df = pd.read_csv(comparison_file)

random.seed(42)

judge_results = []

print("\n🤖 Running LLM judge on 20 Qwen examples...\n")


# ------------------------------------------------------------
# 4. Judge each baseline vs fine-tuned pair
# ------------------------------------------------------------

for _, row in df.iterrows():

    # Randomize A/B position to reduce bias
    if random.random() < 0.5:

        candidate_a = row["baseline_prediction"]
        candidate_b = row["prediction"]

        a_type = "baseline"
        b_type = "finetuned"

    else:

        candidate_a = row["prediction"]
        candidate_b = row["baseline_prediction"]

        a_type = "finetuned"
        b_type = "baseline"

    prompt = f"""
You are evaluating two predicted cluster labels for customer support tickets.

REFERENCE LABEL:
{row["expected"]}

CANDIDATE A:
{candidate_a}

CANDIDATE B:
{candidate_b}

Judge which candidate is better based on:

1. Semantic similarity to the reference
2. Specificity to the support-ticket theme
3. Conciseness
4. Whether it works as a clean cluster label
5. Whether it follows the intended 5-15 word label style

Return ONLY valid JSON:

{{
  "winner": "A",
  "a_score": 1,
  "b_score": 1,
  "reason": "short reason"
}}

winner must be exactly A, B, or TIE.

Scores must be integers from 1 to 5.
"""

    response = client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=180,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    text = response.content[0].text.strip()

    try:

        start = text.index("{")
        end = text.rindex("}") + 1

        result = json.loads(
            text[start:end]
        )

        winner = result["winner"].upper()

        if winner == "A":
            actual_winner = a_type

        elif winner == "B":
            actual_winner = b_type

        else:
            actual_winner = "tie"

        judge_results.append({
            "example": int(row["example"]),
            "expected": row["expected"],
            "baseline_prediction": row["baseline_prediction"],
            "finetuned_prediction": row["prediction"],
            "winner": actual_winner,

            "baseline_score":
                result["a_score"]
                if a_type == "baseline"
                else result["b_score"],

            "finetuned_score":
                result["a_score"]
                if a_type == "finetuned"
                else result["b_score"],

            "reason": result["reason"],
        })

        print(
            f'{int(row["example"]):02d}/20 ✅ '
            f'Winner: {actual_winner}'
        )

    except Exception:
        print(
            f'{int(row["example"]):02d}/20 ❌ '
            f'Parse error: {text}'
        )

    time.sleep(0.2)


# ------------------------------------------------------------
# 5. Save results
# ------------------------------------------------------------

judge_df = pd.DataFrame(judge_results)

SAVE_FILE = RUN / "llm_judge_results.csv"

judge_df.to_csv(
    SAVE_FILE,
    index=False
)

finetuned_wins = (
    judge_df["winner"] == "finetuned"
).sum()

baseline_wins = (
    judge_df["winner"] == "baseline"
).sum()

ties = (
    judge_df["winner"] == "tie"
).sum()

baseline_score = (
    judge_df["baseline_score"].mean()
)

finetuned_score = (
    judge_df["finetuned_score"].mean()
)


# ------------------------------------------------------------
# 6. Final summary
# ------------------------------------------------------------

print("\n==============================")
print("QWEN LLM JUDGE RESULTS")
print("==============================")

print("Fine-tuned wins:", finetuned_wins)
print("Baseline wins:", baseline_wins)
print("Ties:", ties)

print(
    "Baseline average score:",
    round(baseline_score, 3)
)

print(
    "Fine-tuned average score:",
    round(finetuned_score, 3)
)

print("\n✅ Saved:")
print(SAVE_FILE)

display(judge_df)

TimeoutException: Requesting secret ANTHROPIC_API_KEY timed out. Secrets can only be fetched when running from the Colab UI.

In [14]:
# RETRY ANTHROPIC SECRET

import os
import time
from google.colab import userdata

ANTHROPIC_API_KEY = None

for attempt in range(3):
    try:
        ANTHROPIC_API_KEY = userdata.get("ANTHROPIC_API_KEY")
        if ANTHROPIC_API_KEY:
            break
    except Exception as e:
        print(f"Attempt {attempt + 1}/3:", type(e).__name__)
        time.sleep(2)

if ANTHROPIC_API_KEY:
    os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY
    print("✅ ANTHROPIC_API_KEY loaded successfully")
else:
    print("❌ Could not load ANTHROPIC_API_KEY")

✅ ANTHROPIC_API_KEY loaded successfully


In [15]:
# CELL 11 — QWEN LLM-AS-A-JUDGE

import sys
import subprocess
import os
import json
import random
import time
import pandas as pd
from pathlib import Path

# Install Anthropic SDK if needed
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "anthropic"],
    check=True
)

from anthropic import Anthropic

# ------------------------------------------------------------
# API key
# ------------------------------------------------------------

ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")

if not ANTHROPIC_API_KEY:
    raise RuntimeError(
        "ANTHROPIC_API_KEY is not loaded"
    )

client = Anthropic(
    api_key=ANTHROPIC_API_KEY
)

print("✅ Anthropic API ready")


# ------------------------------------------------------------
# Load comparison results
# ------------------------------------------------------------

ROOT = Path("/content/drive/MyDrive/slm-distillation")
RUN = ROOT / "outputs" / "qwen25_3b"

comparison_file = RUN / "baseline_vs_finetuned_comparison.csv"

df = pd.read_csv(comparison_file)

random.seed(42)

judge_results = []

print("\n🤖 Running LLM judge on 20 Qwen examples...\n")


# ------------------------------------------------------------
# Judge each example
# ------------------------------------------------------------

for _, row in df.iterrows():

    # Randomize A/B order
    if random.random() < 0.5:
        candidate_a = row["baseline_prediction"]
        candidate_b = row["prediction"]

        a_type = "baseline"
        b_type = "finetuned"

    else:
        candidate_a = row["prediction"]
        candidate_b = row["baseline_prediction"]

        a_type = "finetuned"
        b_type = "baseline"

    prompt = f"""
You are evaluating two predicted cluster labels for customer support tickets.

REFERENCE LABEL:
{row["expected"]}

CANDIDATE A:
{candidate_a}

CANDIDATE B:
{candidate_b}

Judge which candidate is better based on:

1. Semantic similarity to the reference
2. Specificity to the support-ticket theme
3. Conciseness
4. Whether it works as a clean cluster label
5. Whether it follows the intended 5-15 word label style

Return ONLY valid JSON:

{{
  "winner": "A",
  "a_score": 1,
  "b_score": 1,
  "reason": "short reason"
}}

winner must be exactly A, B, or TIE.

Scores must be integers from 1 to 5.
"""

    response = client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=180,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    text = response.content[0].text.strip()

    try:
        start = text.index("{")
        end = text.rindex("}") + 1

        result = json.loads(
            text[start:end]
        )

        winner = result["winner"].upper()

        if winner == "A":
            actual_winner = a_type

        elif winner == "B":
            actual_winner = b_type

        else:
            actual_winner = "tie"

        judge_results.append({
            "example": int(row["example"]),
            "expected": row["expected"],
            "baseline_prediction": row["baseline_prediction"],
            "finetuned_prediction": row["prediction"],
            "winner": actual_winner,

            "baseline_score":
                result["a_score"]
                if a_type == "baseline"
                else result["b_score"],

            "finetuned_score":
                result["a_score"]
                if a_type == "finetuned"
                else result["b_score"],

            "reason": result["reason"],
        })

        print(
            f'{int(row["example"]):02d}/20 ✅ '
            f'Winner: {actual_winner}'
        )

    except Exception:
        print(
            f'{int(row["example"]):02d}/20 ❌ '
            f'Parse error: {text}'
        )

    time.sleep(0.2)


# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------

judge_df = pd.DataFrame(judge_results)

SAVE_FILE = RUN / "llm_judge_results.csv"

judge_df.to_csv(
    SAVE_FILE,
    index=False
)

finetuned_wins = (
    judge_df["winner"] == "finetuned"
).sum()

baseline_wins = (
    judge_df["winner"] == "baseline"
).sum()

ties = (
    judge_df["winner"] == "tie"
).sum()

baseline_score = (
    judge_df["baseline_score"].mean()
)

finetuned_score = (
    judge_df["finetuned_score"].mean()
)


# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------

print("\n==============================")
print("QWEN LLM JUDGE RESULTS")
print("==============================")

print("Fine-tuned wins:", finetuned_wins)
print("Baseline wins:", baseline_wins)
print("Ties:", ties)

print(
    "Baseline average score:",
    round(baseline_score, 3)
)

print(
    "Fine-tuned average score:",
    round(finetuned_score, 3)
)

print("\n✅ Saved:")
print(SAVE_FILE)

display(judge_df)

✅ Anthropic API ready

🤖 Running LLM judge on 20 Qwen examples...

01/20 ✅ Winner: finetuned
02/20 ✅ Winner: baseline
03/20 ✅ Winner: baseline
04/20 ✅ Winner: finetuned
05/20 ✅ Winner: finetuned
06/20 ✅ Winner: finetuned
07/20 ✅ Winner: finetuned
08/20 ✅ Winner: finetuned
09/20 ✅ Winner: finetuned
10/20 ✅ Winner: finetuned
11/20 ✅ Winner: baseline
12/20 ✅ Winner: finetuned
13/20 ✅ Winner: baseline
14/20 ✅ Winner: baseline
15/20 ✅ Winner: finetuned
16/20 ✅ Winner: finetuned
17/20 ✅ Winner: finetuned
18/20 ✅ Winner: finetuned
19/20 ✅ Winner: finetuned
20/20 ✅ Winner: finetuned

QWEN LLM JUDGE RESULTS
Fine-tuned wins: 15
Baseline wins: 5
Ties: 0
Baseline average score: 2.85
Fine-tuned average score: 3.8

✅ Saved:
/content/drive/MyDrive/slm-distillation/outputs/qwen25_3b/llm_judge_results.csv


,example,expected,baseline_prediction,finetuned_prediction,winner,baseline_score,finetuned_score,reason
0,1,Inquiries about available shipping and deliver...,Shipment Options Inquiry,Customer seeking information on available ship...,finetuned,3,4,Candidate A has higher semantic similarity to ...
1,2,Inquiry about available shipping and delivery ...,Shipment Options Inquiry,Customer seeking information on available ship...,baseline,4,3,Candidate A is more concise (3 words vs 9 word...
2,3,Shipping and Delivery Options Inquiry,Shipment Options Inquiry Requests,Customer Inquiry: Available Shipping and Deliv...,baseline,4,3,Candidate A is more concise (4 words vs 7 word...
3,4,Customers requesting information about availab...,Shipment Options Inquiry Support,Customer seeking information on available ship...,finetuned,3,4,Candidate B has higher semantic similarity to ...
4,5,Shipping and delivery options inquiry,Shipment Options Inquiry,Shipment and Delivery Options Inquiry,finetuned,3,5,Candidate A matches the reference label almost...
5,6,Sign-up process errors and account creation fa...,Notify Sign-Up Issues Errors Problems Inform,Sign-up Assistance and Error Reporting,finetuned,2,4,Candidate A is semantically aligned with the r...
6,7,Sign-up process errors and notification assist...,notify sign-up issues errors problems notify s...,User seeking assistance with reporting sign-up...,finetuned,2,4,"Candidate A is a proper, coherent cluster labe..."
7,8,Sign-up and Registration Error Reporting,Sign-up notification issues,Sign-up Process Assistance and Error Reporting,finetuned,2,4,Candidate B has higher semantic similarity to ...
8,9,Users seeking assistance reporting sign-up and...,Notify sign-up issues errors problems,User seeking assistance to report or inform ab...,finetuned,2,4,"Candidate B maintains semantic similarity, is ..."
9,10,Sign-up registration errors and problem reporting,notify sign-up issues errors problems notify s...,User seeking assistance to report or inform ab...,finetuned,2,4,"B is semantically similar, specific to support..."


In [16]:
# CELL 12 — SAVE FINAL QWEN EXPERIMENT SUMMARY

import json
import pandas as pd
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/slm-distillation")
RUN = ROOT / "outputs" / "qwen25_3b"

comparison = pd.read_csv(
    RUN / "baseline_vs_finetuned_comparison.csv"
)

judge = pd.read_csv(
    RUN / "llm_judge_results.csv"
)

# ------------------------------------------------------------
# Calculate metrics directly from saved files
# ------------------------------------------------------------

baseline_cosine = comparison["baseline_similarity"].mean()
finetuned_cosine = comparison["finetuned_similarity"].mean()
cosine_change = finetuned_cosine - baseline_cosine

finetuned_better = (
    comparison["improvement"] > 0.0001
).sum()

baseline_better = (
    comparison["improvement"] < -0.0001
).sum()

finetuned_wins = (
    judge["winner"] == "finetuned"
).sum()

baseline_wins = (
    judge["winner"] == "baseline"
).sum()

ties = (
    judge["winner"] == "tie"
).sum()

baseline_judge = judge["baseline_score"].mean()
finetuned_judge = judge["finetuned_score"].mean()


# ------------------------------------------------------------
# Markdown summary
# ------------------------------------------------------------

summary = f"""# Qwen2.5-3B-Instruct Benchmark Results

## Model

- Model: `Qwen/Qwen2.5-3B-Instruct`
- Fine-tuning method: 4-bit QLoRA
- GPU: Tesla T4
- LoRA rank: 16
- LoRA alpha: 16
- LoRA dropout: 0.05
- Learning rate: 2e-4
- Max sequence length: 384

## Dataset

- Training examples: 70
- Validation examples: 15
- Test examples: 20
- Prompt variants: P1-P5
- Training epochs: 3

## Training

Training completed successfully.

- Epoch 1 validation loss: 1.5322
- Epoch 2 validation loss: 0.9958
- Epoch 3 validation loss: 0.8789
- Best epoch: 3
- Best checkpoint: checkpoint-27

Validation loss improved across all three epochs.

## Cosine Similarity

- Baseline: {baseline_cosine:.4f}
- Fine-tuned: {finetuned_cosine:.4f}
- Improvement: {cosine_change:+.4f}
- Fine-tuned better on: {finetuned_better}/20
- Baseline better on: {baseline_better}/20

## LLM-as-a-Judge

- Fine-tuned wins: {finetuned_wins}
- Baseline wins: {baseline_wins}
- Ties: {ties}
- Baseline average score: {baseline_judge:.2f}
- Fine-tuned average score: {finetuned_judge:.2f}

## Conclusion

Fine-tuning Qwen2.5-3B produced a clear improvement over the baseline.

The fine-tuned model achieved higher semantic similarity on most test
examples and was strongly preferred by the LLM judge.

Qwen2.5-3B therefore showed a positive response to the current QLoRA
configuration and is a strong candidate for the project.
"""

summary_file = RUN / "FINAL_QWEN_RESULTS.md"

summary_file.write_text(
    summary,
    encoding="utf-8"
)


# ------------------------------------------------------------
# Machine-readable metrics
# ------------------------------------------------------------

final_metrics = {
    "model": "Qwen/Qwen2.5-3B-Instruct",
    "training_status": "successful",

    "best_epoch": 3,
    "best_checkpoint": "checkpoint-27",
    "best_validation_loss": 0.8789467215538025,

    "baseline_cosine_similarity": float(baseline_cosine),
    "finetuned_cosine_similarity": float(finetuned_cosine),
    "cosine_change": float(cosine_change),

    "finetuned_better_examples": int(finetuned_better),
    "baseline_better_examples": int(baseline_better),

    "llm_judge_finetuned_wins": int(finetuned_wins),
    "llm_judge_baseline_wins": int(baseline_wins),
    "llm_judge_ties": int(ties),

    "baseline_judge_score": float(baseline_judge),
    "finetuned_judge_score": float(finetuned_judge),

    "overall_result": "finetuning_improved_performance"
}

metrics_file = RUN / "final_metrics.json"

with open(metrics_file, "w", encoding="utf-8") as f:
    json.dump(final_metrics, f, indent=2)


# ------------------------------------------------------------
# Final output
# ------------------------------------------------------------

print("=" * 60)
print("✅ QWEN EXPERIMENT SAVED")
print("=" * 60)

print("\nBaseline cosine:", round(baseline_cosine, 4))
print("Fine-tuned cosine:", round(finetuned_cosine, 4))
print("Improvement:", round(cosine_change, 4))

print("\nFine-tuned judge wins:", finetuned_wins)
print("Baseline judge wins:", baseline_wins)

print("Baseline judge score:", round(baseline_judge, 2))
print("Fine-tuned judge score:", round(finetuned_judge, 2))

print("\nSummary:")
print(summary_file)

print("\nMetrics:")
print(metrics_file)

print("\n===== SAVED QWEN FILES =====")

for path in sorted(RUN.iterdir()):
    print("✅", path.name)

✅ QWEN EXPERIMENT SAVED

Baseline cosine: 0.7252
Fine-tuned cosine: 0.8077
Improvement: 0.0825

Fine-tuned judge wins: 15
Baseline judge wins: 5
Baseline judge score: 2.85
Fine-tuned judge score: 3.8

Summary:
/content/drive/MyDrive/slm-distillation/outputs/qwen25_3b/FINAL_QWEN_RESULTS.md

Metrics:
/content/drive/MyDrive/slm-distillation/outputs/qwen25_3b/final_metrics.json

===== SAVED QWEN FILES =====
✅ FINAL_QWEN_RESULTS.md
✅ baseline_test_predictions.csv
✅ baseline_vs_finetuned_comparison.csv
✅ experiment_config.json
✅ final_adapter
✅ final_metrics.json
✅ finetuned_test_predictions.csv
✅ llm_judge_results.csv
✅ training
✅ training_metrics.json
✅ training_summary.json
✅ validation_metrics.json


In [17]:
# CELL 13 — QWEN OUTPUT SAFETY AUDIT

from pathlib import Path
import re

ROOT = Path("/content/drive/MyDrive/slm-distillation")
RUN = ROOT / "outputs" / "qwen25_3b"

print("=" * 60)
print("QWEN OUTPUT SAFETY AUDIT")
print("=" * 60)

# ------------------------------------------------------------
# 1. List every saved file + size
# ------------------------------------------------------------

all_files = [
    p for p in RUN.rglob("*")
    if p.is_file()
]

print(f"\nTotal files found: {len(all_files)}\n")

for path in sorted(all_files):
    size_mb = path.stat().st_size / (1024 ** 2)

    relative = path.relative_to(RUN)

    print(
        f"{size_mb:9.2f} MB  |  {relative}"
    )


# ------------------------------------------------------------
# 2. Detect large files that should NEVER go to GitHub
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("LARGE FILES — KEEP IN GOOGLE DRIVE")
print("=" * 60)

large_files = []

for path in all_files:
    size_mb = path.stat().st_size / (1024 ** 2)

    if size_mb > 25:
        large_files.append((path, size_mb))

if large_files:
    for path, size_mb in sorted(
        large_files,
        key=lambda x: x[1],
        reverse=True
    ):
        print(
            f"🚫 {size_mb:.2f} MB | "
            f"{path.relative_to(RUN)}"
        )
else:
    print("✅ No files larger than 25 MB")


# ------------------------------------------------------------
# 3. Secret scan on text-readable files
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("SECRET SCAN")
print("=" * 60)

text_extensions = {
    ".json",
    ".jsonl",
    ".md",
    ".txt",
    ".csv",
    ".py",
    ".ipynb",
    ".yaml",
    ".yml",
}

secret_patterns = [
    r"sk-ant-[A-Za-z0-9_-]{10,}",
    r"sk-[A-Za-z0-9_-]{20,}",
    r"hf_[A-Za-z0-9]{15,}",
    r"ghp_[A-Za-z0-9]{20,}",
    r"github_pat_[A-Za-z0-9_]{20,}",
]

secret_hits = []

for path in all_files:

    if path.suffix.lower() not in text_extensions:
        continue

    try:
        text = path.read_text(
            encoding="utf-8",
            errors="ignore"
        )

        for pattern in secret_patterns:

            matches = re.findall(
                pattern,
                text
            )

            if matches:
                secret_hits.append(
                    (
                        path.relative_to(RUN),
                        pattern,
                        len(matches)
                    )
                )

    except Exception:
        pass


if secret_hits:

    print("❌ POSSIBLE SECRET FOUND")

    for file, pattern, count in secret_hits:
        print(
            f"File: {file} | "
            f"Matches: {count}"
        )

else:
    print("✅ No obvious API keys/tokens detected")


# ------------------------------------------------------------
# 4. Files safe to package for GitHub
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("SMALL RESULT FILES FOR GITHUB")
print("=" * 60)

github_extensions = {
    ".json",
    ".md",
}

github_files = []

for path in all_files:

    if path.suffix.lower() not in github_extensions:
        continue

    size_mb = path.stat().st_size / (1024 ** 2)

    if size_mb < 10:
        github_files.append(path)

for path in sorted(github_files):
    print(
        "✅",
        path.relative_to(RUN)
    )


print("\n" + "=" * 60)
print("AUDIT COMPLETE")
print("=" * 60)

print("\n⚠️ Keep these in Google Drive only:")
print("- final_adapter/")
print("- training/checkpoint-*/")
print("- *.safetensors")
print("- optimizer.pt")
print("- scheduler.pt")
print("- CSV prediction/evaluation files")

print("\n✅ GitHub will only receive:")
print("- notebook")
print("- FINAL_QWEN_RESULTS.md")
print("- experiment_config.json")
print("- final_metrics.json")
print("- training_metrics.json")
print("- training_summary.json")
print("- validation_metrics.json")

QWEN OUTPUT SAFETY AUDIT

Total files found: 65

     0.00 MB  |  FINAL_QWEN_RESULTS.md
     0.00 MB  |  baseline_test_predictions.csv
     0.00 MB  |  baseline_vs_finetuned_comparison.csv
     0.00 MB  |  experiment_config.json
     0.00 MB  |  final_adapter/README.md
     0.00 MB  |  final_adapter/adapter_config.json
   114.25 MB  |  final_adapter/adapter_model.safetensors
     0.00 MB  |  final_adapter/added_tokens.json
     0.00 MB  |  final_adapter/chat_template.jinja
     1.59 MB  |  final_adapter/merges.txt
     0.00 MB  |  final_adapter/special_tokens_map.json
    10.89 MB  |  final_adapter/tokenizer.json
     0.00 MB  |  final_adapter/tokenizer_config.json
     2.65 MB  |  final_adapter/vocab.json
     0.00 MB  |  final_metrics.json
     0.00 MB  |  finetuned_test_predictions.csv
     0.01 MB  |  llm_judge_results.csv
     0.00 MB  |  training/checkpoint-18/README.md
     0.00 MB  |  training/checkpoint-18/adapter_config.json
   114.25 MB  |  training/checkpoint-18/adapter_mod

In [18]:
# CELL 14 — FIND + AUDIT QWEN NOTEBOOK

from pathlib import Path
import json
import re

print("=" * 60)
print("QWEN NOTEBOOK SAFETY CHECK")
print("=" * 60)

# ------------------------------------------------------------
# 1. Find Qwen notebook
# ------------------------------------------------------------

search_roots = [
    Path("/content/drive/MyDrive"),
    Path("/content"),
]

matches = []

for root in search_roots:
    if root.exists():
        for p in root.rglob("*.ipynb"):
            name = p.name.lower()

            if (
                "qwen" in name
                and "benchmark" in name
            ):
                matches.append(p)

# Remove duplicates
matches = list(dict.fromkeys(matches))

if not matches:
    print("\n❌ Could not find qwen benchmark notebook.")
    print("Save the notebook to Google Drive first.")
else:
    print("\nFound notebook(s):")

    for i, p in enumerate(matches, 1):
        size_mb = p.stat().st_size / (1024 ** 2)
        print(f"{i}. {p}")
        print(f"   Size: {size_mb:.2f} MB")

    # Use most recently modified matching notebook
    NOTEBOOK = max(
        matches,
        key=lambda p: p.stat().st_mtime
    )

    print("\n✅ Selected notebook:")
    print(NOTEBOOK)


    # ------------------------------------------------------------
    # 2. Validate notebook JSON
    # ------------------------------------------------------------

    with open(NOTEBOOK, "r", encoding="utf-8") as f:
        nb = json.load(f)

    cells = nb.get("cells", [])

    code_cells = [
        c for c in cells
        if c.get("cell_type") == "code"
    ]

    markdown_cells = [
        c for c in cells
        if c.get("cell_type") == "markdown"
    ]

    print("\n===== NOTEBOOK STRUCTURE =====")
    print("Total cells:", len(cells))
    print("Code cells:", len(code_cells))
    print("Markdown cells:", len(markdown_cells))
    print("✅ Notebook JSON is valid")


    # ------------------------------------------------------------
    # 3. Scan notebook source + outputs for secrets
    # ------------------------------------------------------------

    raw_text = NOTEBOOK.read_text(
        encoding="utf-8",
        errors="ignore"
    )

    secret_patterns = {
        "Anthropic key": r"sk-ant-[A-Za-z0-9_-]{10,}",
        "OpenAI-style key": r"sk-[A-Za-z0-9_-]{20,}",
        "Hugging Face token": r"hf_[A-Za-z0-9]{15,}",
        "GitHub classic PAT": r"ghp_[A-Za-z0-9]{20,}",
        "GitHub PAT": r"github_pat_[A-Za-z0-9_]{20,}",
    }

    hits = []

    for label, pattern in secret_patterns.items():
        found = re.findall(pattern, raw_text)

        if found:
            hits.append(
                (label, len(found))
            )

    print("\n===== SECRET SCAN =====")

    if hits:
        print("❌ POSSIBLE SECRET FOUND")

        for label, count in hits:
            print(
                f"{label}: {count} possible match(es)"
            )

    else:
        print("✅ No obvious API keys/tokens detected")


    # ------------------------------------------------------------
    # 4. Check for expected experiment content
    # ------------------------------------------------------------

    expected_terms = [
        "Qwen/Qwen2.5-3B-Instruct",
        "baseline_test_predictions.csv",
        "finetuned_test_predictions.csv",
        "checkpoint-27",
        "llm_judge_results.csv",
        "FINAL_QWEN_RESULTS.md",
    ]

    print("\n===== EXPERIMENT CONTENT CHECK =====")

    for term in expected_terms:

        if term in raw_text:
            print("✅", term)
        else:
            print("⚠️ Missing:", term)


    # ------------------------------------------------------------
    # Final
    # ------------------------------------------------------------

    print("\n" + "=" * 60)
    print("NOTEBOOK AUDIT COMPLETE")
    print("=" * 60)

    print("\nNotebook path:")
    print(NOTEBOOK)

    print(
        "\nNotebook size:",
        round(
            NOTEBOOK.stat().st_size / (1024 ** 2),
            2
        ),
        "MB"
    )

QWEN NOTEBOOK SAFETY CHECK

Found notebook(s):
1. /content/drive/MyDrive/Colab Notebooks/qwen25_3b_benchmark.ipynb
   Size: 0.33 MB

✅ Selected notebook:
/content/drive/MyDrive/Colab Notebooks/qwen25_3b_benchmark.ipynb

===== NOTEBOOK STRUCTURE =====
Total cells: 18
Code cells: 17
Markdown cells: 1
✅ Notebook JSON is valid

===== SECRET SCAN =====
✅ No obvious API keys/tokens detected

===== EXPERIMENT CONTENT CHECK =====
✅ Qwen/Qwen2.5-3B-Instruct
✅ baseline_test_predictions.csv
✅ finetuned_test_predictions.csv
✅ checkpoint-27
✅ llm_judge_results.csv
✅ FINAL_QWEN_RESULTS.md

NOTEBOOK AUDIT COMPLETE

Notebook path:
/content/drive/MyDrive/Colab Notebooks/qwen25_3b_benchmark.ipynb

Notebook size: 0.33 MB
